# Sign-free ($h_z$) at **L=8** — **v1**: CNN Block-1 + Wilson + N-layer transformer Block-3

Can a deeper v1 transformer **match the CNN's accuracy** at the first NQS-only size (L=8, 112 qubits)?
Sweep $h_z \in \{0.25, 0.30\}$ ($h_x=h_y=0$, **real** $\psi$).

- **Block-1 = the *same* non-invariant CNN as the CNN baseline**; **Wilson** unchanged. Only **Block-3**
  is the transformer, with **`TF_LAYERS` a knob** (default **3**; bump to **4**). No complex readout
  (sign-free ⇒ real ansatz).
- **Why this test:** at L=8 the 2-layer `v1c8`/`v1c16` reached V-score ~2e-4 vs the CNN's ~1e-5; the
  earlier 6e-5-diag_shift `tf` runs *diverged* here. This asks whether **more depth (3–4 layers, d=16)
  at a stable `diag_shift`** closes that gap.

**Baselines** (committed): the CNN phase-sweep run (`..._hz0.250/0.300.json`, no arm tag) and the
2-layer **`v1c16`** (`..._hz0.25_v1c16.json`). No ED at L=8 — rank by energy/V-score. Live to **W&B online**.
**Resumable** via Drive + skip-if-complete.


## Before you start
1. **Runtime → GPU** (A100/L4 preferred; L=8 real is ~20 s/step on A100).
2. **GitHub token** + **W&B API key** ready.
3. Run top-to-bottom; restart after install if prompted, then resume from *Mount Google Drive*.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned stack

`requirements.txt` pins a **CPU** `jaxlib` (for the NERSC conda env); on Colab we install the
**CUDA 12** build of the same `jax==0.5.2` instead, plus `netket==3.16.1.post1`, `flax==0.10.4`, `wandb`.

⚠️ If pip changes the `jaxlib` already loaded in this session, Colab will ask you to **restart the
runtime** — do it, then resume from the *Mount Google Drive* cell.

In [ ]:
%pip install -q "jax[cuda12]==0.5.2" "jaxlib==0.5.1" netket==3.16.1.post1 flax==0.10.4 wandb
print("\nInstall finished. If a 'restart runtime' notice appeared, do Runtime > Restart session,")
print("then continue from the 'Mount Google Drive' cell (do NOT re-run this install cell).")

In [ ]:
import jax, jaxlib, netket, flax
print("jax", jax.__version__, "| jaxlib", jaxlib.__version__,
      "| netket", netket.__version__, "| flax", flax.__version__)
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), \
    "No GPU visible to JAX — set Runtime>GPU and/or re-check the jax[cuda12] install."
print("OK: GPU visible.")

## 3. Mount Google Drive  *(resume point after any runtime restart)*
Results and the cloned repo live under `MyDrive/2D-TC-colab/` so they survive disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone (or update) the repo into Drive
Private repo → paste a GitHub token when prompted. It is used only for the clone, then stripped from
the stored remote (not saved to Drive).

In [ ]:
import os, getpass, subprocess

DRIVE_ROOT = "/content/drive/MyDrive/2D-TC-colab"   # persists across sessions
BRANCH     = "v3-complex-readout"                    # has the transformer code + CNN baselines
REPO_DIR   = os.path.join(DRIVE_ROOT, "2D-TC")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    tok = getpass.getpass("GitHub token (repo read access): ")
    url = f"https://{tok}@github.com/SanzharBissenali/2D-TC.git"
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    "https://github.com/SanzharBissenali/2D-TC.git"], check=True)
    del tok, url
else:
    print("repo already present; trying a fast-forward pull (harmless if it fails)")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "| branch:", head)
assert os.path.exists("model/transformer_block.py"), "transformer code missing — wrong branch?"
print("OK: transformer code present.")

## 5. Log in to Weights & Biases (online)
Key from [wandb.ai/authorize](https://wandb.ai/authorize). Stored in `~/.netrc`, which the training
subprocesses inherit, so runs stream **live** to the dashboard.

In [ ]:
import os, getpass, wandb

os.environ["WANDB_MODE"] = "online"          # Colab has internet -> truly live
wandb.login(key=getpass.getpass("W&B API key (wandb.ai/authorize): "))

WANDB_PROJECT = "2d-tc"
WANDB_ENTITY  = ""     # optional: your wandb team/username; "" = your default entity
print("W&B online. project =", WANDB_PROJECT, "| entity =", WANDB_ENTITY or "(default)")

## 6. Sweep configuration
**`TF_LAYERS`** and **`D_MODEL`** are the capacity knobs. **`DIAG_SHIFT`** is the stability/accuracy
knob: `3e-4` was rock-stable at L=8 for the 2-layer arm; the deeper model + more steps is the bet to
close the gap. If it converges cleanly, try `1e-4` for a deeper minimum (watch for V-score blow-ups).

In [ ]:
D_MODEL    = 16
TF_LAYERS  = 3       # <<< KNOB: transformer blocks AFTER Wilson (try 4)
TF_HEADS   = 2
LX_LIST    = [8]
HZ_LIST    = [0.25, 0.30]
SEED       = 0
SIM_TIME   = 3.0                 # 300 steps
DIAG_SHIFT = "3e-4"              # <<< stability knob (try "1e-4" for a deeper minimum)

ARM = f"v1r{D_MODEL}n{TF_LAYERS}"                            # v1r16n3 (r = real, sign-free)
ARM_FLAGS = (
    f"--symmetric_block transformer "
    f"--tf_layers {TF_LAYERS} --tf_dmodel {D_MODEL} --tf_heads {TF_HEADS} "
    f"--tf_ffn_mult 2 --tf_activation relu "
    f"--diag_shift {DIAG_SHIFT} --lr_schedule cosine --lr_final_frac 0.25"
)
WANDB_GROUP = f"hz_{ARM}"
print("arm:", ARM, "|", len(LX_LIST) * len(HZ_LIST), "points | flags:", ARM_FLAGS)

## 7. Run the sweep (resumable, live to W&B)
L=8 ≥ 6 ⇒ end observables (Wilson/Rényi) run too, so `is_complete` requires them — a walltime kill
mid-observables re-runs that point.

In [ ]:
import os, subprocess, sys

REPO   = os.getcwd()
OUTDIR = os.path.join(REPO, "results", "nqs")
os.makedirs(OUTDIR, exist_ok=True)
env = dict(os.environ, PYTHONPATH=REPO, WANDB_MODE="online", WANDB_DIR=REPO)

def is_complete(base, L):
    return subprocess.run([sys.executable, os.path.join(REPO, "scripts", "is_complete.py"),
                           base, str(L)]).returncode == 0

for L in LX_LIST:
    for hz in HZ_LIST:
        jobid = f"L{L}_hx0.00_hz{hz:.2f}_{ARM}"
        base  = os.path.join(OUTDIR, f"G-equiv_1_{jobid}")
        if is_complete(base, L):
            print(f"== skip {jobid} (complete) =="); continue
        print(f"\n{'='*70}\n RUN {jobid}   (sim_time={SIM_TIME}, diag_shift={DIAG_SHIFT})\n{'='*70}", flush=True)
        entity = f" --wandb_entity {WANDB_ENTITY}" if WANDB_ENTITY else ""
        cmd = (
            f"cd {OUTDIR} && PYTHONPATH={REPO} python {REPO}/main.py "
            f"--outindex 1 --jobid {jobid} "
            f"--Lx {L} --hx 0.0 --hy 0.0 --hz {hz:.2f} "
            f"--dt 0.01 --diag_shift 6e-5 --sim_time {SIM_TIME} --seed {SEED} "
            f"--architecture Combo --channels_noninv 1,16 --channels_inv 16,8,1 --kernel_size 2 "
            f"--n_samples_fin 8192 --use_custom_sampler "
            f"{ARM_FLAGS} "
            f"--wandb --wandb_project {WANDB_PROJECT} --wandb_group {WANDB_GROUP}{entity}"
        )
        rc = subprocess.run(["bash", "-lc", cmd], env=env).returncode
        print(f"===== {jobid} exit {rc} =====", flush=True)

print("\nSweep pass complete. Re-run to resume any unfinished points.")

## 8. Analysis — does the deeper v1 match CNN at L=8?
No ED at L=8, so we rank by energy (lower = better) and V-score, against the CNN baseline and the
2-layer `v1c16`. `dE_CNN = E_arm − E_CNN` (negative = beats CNN); the CNN's V-score is the target to match.

In [ ]:
import json, os, statistics as st
import numpy as np, matplotlib.pyplot as plt

OUTDIR = os.path.join(os.getcwd(), "results", "nqs"); TAIL = 10
def _c(x):
    return complex(x) if not isinstance(x, dict) else complex(x.get("real", np.nan), x.get("imag", 0.0))
def load(tag):
    p = os.path.join(OUTDIR, f"G-equiv_1_{tag}.json")
    if not os.path.exists(p): return None
    e = [_c(x).real for x in json.load(open(p)).get("energy", [])]
    v = [_c(x).real for x in json.load(open(p)).get("Vscore", [])]
    return None if not e else dict(E=st.median(e[-TAIL:]), V=(st.median(v[-TAIL:]) if v else np.nan))

L = 8
print(f"=== L={L} sign-free:  {ARM} (3-layer) vs CNN baseline vs v1c16 (2-layer) ===")
print(f"{'hz':>5} {'E_cnn':>13} {'E_'+ARM:>14} {'dE_CNN':>10} {'V_cnn':>9} {'V_arm':>9} {'V_v1c16':>9}")
for hz in HZ_LIST:
    c  = load(f"L{L}_hx0.00_hz{hz:.3f}")          # untagged CNN phase-sweep baseline (3 decimals!)
    t  = load(f"L{L}_hx0.00_hz{hz:.2f}_{ARM}")
    v2 = load(f"L{L}_hx0.00_hz{hz:.2f}_v1c16")    # 2-layer d=16 reference
    if not c: print(f"{hz:>5.2f}  (no CNN baseline)"); continue
    if not t: print(f"{hz:>5.2f} {c['E']:>13.4f} {'(pending)':>14}"); continue
    dE = t["E"] - c["E"]; vv = f"{v2['V']:>9.1e}" if v2 else f"{'--':>9}"
    print(f"{hz:>5.2f} {c['E']:>13.4f} {t['E']:>14.4f} {dE:>+10.4f} {c['V']:>9.1e} {t['V']:>9.1e} {vv}")

# V-score bar comparison per hz (log scale): CNN target vs 2-layer vs this deeper arm
fig, ax = plt.subplots(1, len(HZ_LIST), figsize=(5 * len(HZ_LIST), 4), squeeze=False)
for i, hz in enumerate(HZ_LIST):
    c  = load(f"L{L}_hx0.00_hz{hz:.3f}"); t = load(f"L{L}_hx0.00_hz{hz:.2f}_{ARM}")
    v2 = load(f"L{L}_hx0.00_hz{hz:.2f}_v1c16")
    labels, vals = [], []
    for name, r in [("CNN", c), ("v1c16(2L)", v2), (ARM, t)]:
        if r: labels.append(name); vals.append(r["V"])
    a = ax[0, i]; a.bar(labels, vals, color=["gray", "steelblue", "crimson"][:len(vals)])
    a.set(title=f"L=8, hz={hz}: V-score (lower=better)", yscale="log", ylabel="V-score")
fig.tight_layout(); plt.show()

## Runtime notes
- **L=8 real** is ~20 s/step on an A100 (slower on T4). 300 steps ⇒ ~1.5–2 h/point + end observables
  (Wilson/Rényi over 2^112 via sampling). 2 hz points ⇒ plan a few hours; resumable across sessions.
- If clean and fast, add hz points (`0.15, 0.35, …`) or bump `TF_LAYERS=4`. If it diverges (V-score→1),
  raise `DIAG_SHIFT` to `1e-3`.
- Watch live on wandb.ai (group `hz_<arm>`). Results in `MyDrive/2D-TC-colab/2D-TC/results/nqs/`.
